Implementation of Query Decomposition

In [6]:
import os
from dotenv import load_dotenv
from langchain.document_loaders import TextLoader
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableSequence
from  langchain.chat_models import init_chat_model

In [7]:
# Load data
# Load and split data

loader = TextLoader('sample.txt')
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

In [8]:
# Embed the document and Create vectorstore
embedding_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(chunks, embedding_model)

#Create MMR Retriever
retriever = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 5}
)
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002436FAF3110>, search_type='mmr', search_kwargs={'k': 5})

In [14]:
load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

llm = init_chat_model(model='groq:llama-3.3-70b-versatile')

llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000024318149710>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002431A012FD0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [15]:
# Query decomposition
decomposition_prompt = PromptTemplate.from_template(
    """
    You are an AI assistant. Decompose the following complext into 2 to 4 smaller sub-questions for improve document retriever.
    
    Question: '{question}'
    
    Sub-questions:
    """
)

decompose_chain = decomposition_prompt | llm | StrOutputParser()

In [16]:
query = 'How does LangChain use memory and agents compared to CrewAI?'

decomposition_question = decompose_chain.invoke({'question': query})

In [17]:
print(decomposition_question)

To improve document retrieval, I can decompose the complex question into the following 3 smaller sub-questions:

1. **What is LangChain's approach to memory management?** 
   This sub-question focuses on understanding how LangChain utilizes memory, which is essential for its functionality.

2. **How does LangChain implement and utilize agents?** 
   This sub-question delves into the specifics of LangChain's agent architecture, aiming to clarify its capabilities and limitations.

3. **What are the key differences between LangChain and CrewAI in terms of memory and agent usage?** 
   This comparative sub-question seeks to highlight the distinctions between the two systems, allowing for a more nuanced understanding of their respective strengths and weaknesses.

By addressing these sub-questions, we can gather more specific and detailed information about LangChain and CrewAI, facilitating a more effective comparison and analysis of their approaches to memory and agent utilization.


In [18]:
# QA chain per sub-question

qa_prompt = PromptTemplate.from_template(
    '''
    Use the context below to answer the question.
    
    Context: {context}
    
    Question: {input}
    '''
)

qa_chain = create_stuff_documents_chain(llm=llm, prompt=qa_prompt)

In [21]:
# Full RAG pipeline 

def full_query_decomposition_rag_pipeline(user_query):
    
    # Decompse the query
    sub_qs_text = decompose_chain.invoke({'question': user_query})
    sub_questions = [
        q.strip('-1234567890.') for q in sub_qs_text.split('\n') if q.strip()
    ]
    
    results = []
    for sub_question in sub_questions:
        docs = retriever.invoke(sub_question)
        result = qa_chain.invoke({'input': sub_question, 'context': docs})
        
        results.append(f'Q: {sub_question}\nA: {result}')
        
    
    return '\n\n'.join(results)

In [22]:
# Run
query = 'How does Langchain use memory and agents compared to CrewAI?'

final_answer = full_query_decomposition_rag_pipeline(query)
print('Final Result:\n')
print(final_answer)

Final Result:

Q: To improve document retrieval, I'll decompose the complex question into 3 smaller sub-questions:
A: To improve document retrieval, let's break down the complex question into three smaller sub-questions:

1. What role does BM25 play in a hybrid search system, and how does it contribute to retrieval accuracy?
2. How can LangGraph's capabilities enhance the document retrieval process, particularly in terms of transparency and persistence?
3. In what ways do FAISS and LLMs work together to enable efficient and accurate document retrieval, especially in high-dimensional spaces and with large datasets?

Q:  **What is Langchain's approach to memory management?** 
A: LangChain's approach to memory management involves a dedicated module that stores and retrieves information from past interactions, allowing conversational AI to maintain context and remember user preferences over multiple turns. This module is specifically designed to enable the model to treat interactions as pa